# Module 6.5 — Re-ranking Strategies

**Two-stage retrieval**:
1. **Stage 1 — Recall**: retrieve a large candidate set (e.g. top-20) quickly
2. **Stage 2 — Precision**: re-rank candidates with a powerful cross-encoder

Cross-encoders are slower but far more accurate than bi-encoders for ranking.

In [ ]:
# !pip install sentence-transformers cohere

# ── Option A: Local cross-encoder re-ranker (free) ────────────────────────────
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

docs = [
    Document(page_content="The transformer architecture uses self-attention mechanisms."),
    Document(page_content="BERT is a bidirectional transformer model pre-trained on masked language modelling."),
    Document(page_content="GPT uses a decoder-only transformer for autoregressive generation."),
    Document(page_content="Attention is all you need — the seminal transformer paper by Vaswani et al."),
    Document(page_content="Recurrent neural networks process sequences step by step."),
    Document(page_content="Convolutional networks are mainly used for image classification."),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="rerank_demo")
query      = "How does the transformer attention mechanism work?"

# Stage 1: retrieve top-6 candidates
candidates = vs.similarity_search(query, k=6)
print("Stage 1 — Initial retrieval order:")
for i, d in enumerate(candidates, 1):
    print(f"  [{i}] {d.page_content[:70]}")

# Stage 2: re-rank with cross-encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
pairs  = [(query, d.page_content) for d in candidates]
scores = cross_encoder.predict(pairs)

reranked = sorted(zip(scores, candidates), reverse=True)
print("\nStage 2 — After cross-encoder re-ranking:")
for score, doc in reranked:
    print(f"  [{score:6.3f}] {doc.page_content[:70]}")


In [ ]:
# ── Option B: Cohere Rerank (cloud) ──────────────────────────────────────────
# !pip install cohere langchain-cohere
# import cohere, os
# co = cohere.Client(os.environ["COHERE_API_KEY"])
#
# results = co.rerank(
#     query=query,
#     documents=[d.page_content for d in candidates],
#     model="rerank-english-v3.0",
#     top_n=3,
# )
# print("Cohere Rerank results:")
# for r in results.results:
#     print(f"  [{r.relevance_score:.4f}] {candidates[r.index].page_content[:70]}")

print("Uncomment the block above and set COHERE_API_KEY to use Cohere Rerank.")
